### Needs `01-ReferenceDFs.ipynb` to be executed first

Reads from:
- `PipelineB_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
import random

## Import Data

In [ ]:
# As this was the first evaluation script of this kind, I kept the non-ynom analysis for reasoning why we moved to ynorm as the standard
gs_extractions_raw  = pd.read_json("PipelineB.json")
gs_extractions_raw_ynorm = pd.read_json("PipelineB_ynorm.json")
CATEGORIES = ["value", "unit"]
VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2", "g"]

#### Cleanup Import

In [ ]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    gs_extractions_raw[col] = gs_extractions_raw[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    gs_extractions_raw_ynorm[col] = gs_extractions_raw_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [ ]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set


def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v:<2}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [ ]:
just_eval, expended = eval(gs_extractions_raw, False)

print_eval(just_eval)

### Exact

In [ ]:
just_eval_exact, _ = eval(gs_extractions_raw, True)

print_eval(just_eval_exact)

## YNROM

In [ ]:
ynorm, expended_ynorm = eval(gs_extractions_raw_ynorm, False)

print_eval(ynorm)

### Exact

In [ ]:
ynorm_exact, forGEPA = eval(gs_extractions_raw_ynorm, True)

print_eval(ynorm_exact)

# Detailed Comparison

In [ ]:
def save_eval_comparison(eval_raw, eval_ynorm, filepath="eval_comparison.json"):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    df_results.to_json(filepath, orient="records", indent=2)
    df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = save_eval_comparison(just_eval, ynorm)
print(comparison)


In [ ]:
comparison_exact = save_eval_comparison(just_eval_exact, ynorm_exact)
print(comparison_exact)

In [ ]:
# No merge necessary as the two DFs have the same index
comparison_exact["raw_true_any"] = comparison["raw_true"]
comparison_exact["fewer_hits_raw"] = comparison_exact["raw_true"] - comparison_exact["raw_true_any"]
comparison_exact["ynorm_true_any"] = comparison["ynorm_true"]
comparison_exact["fewer_hits_ynorm"] = comparison_exact["ynorm_true"] - comparison_exact["ynorm_true_any"]
comparison_exact[["variant","category","raw_true","ynorm_true","raw_true_any","fewer_hits_raw","ynorm_true_any","fewer_hits_ynorm"]]

* You can see that the hits have dropped in general. Interestingly not all equal, not even for the same model.
* The drops are more evenly distributed for units than for values.


# Stuff for GEPA

In [ ]:
just_i = gs_extractions_raw_ynorm[
    ['report_name',
    'status',
    'scopes_present',
    'years_present',
    'year',
    'scope',
    'page',
    'value',
    'unit',
    'unit_normalized',
    'label',
    'value_i1',
    'value_i2',
    'unit_i1',
    'unit_i2',
    'label_i1',
    'label_i2']
].copy()

In [ ]:
VARIANTS = ["i1", "i2"] # Just for this part, to only get the intr. cases
# Exact comparison!

eval_i, eval_i_expended = eval(just_i, True)

print_eval(eval_i)

VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2", "g"] # switching back to original values

### Choosing 60% Dataset for GEPA with Tinking-Model

In [ ]:
forGEPA = forGEPA[["report_name","t1_value_hit","t2_value_hit"]]
forGEPA

In [ ]:
t1 = forGEPA["t1_value_hit"]
t2 = forGEPA["t2_value_hit"]

not_match = (t1 != t2).sum()
t1_true_t2_false = (t1 & ~t2).sum()
t1_false_t2_true = (~t1 & t2).sum()

print(f"t1 vs t2 not matching: {not_match}")
print(f"t1=True, t2=False: {t1_true_t2_false}")
print(f"t1=False, t2=True: {t1_false_t2_true}")

==> Seems pretty much 50/50 so no run was really better or worse. Therefore, dropping t2.

In [ ]:
forGEPA = forGEPA[["report_name","t1_value_hit"]]

In [ ]:
hits = forGEPA.groupby("report_name")["t1_value_hit"].all()
GEPA_TRUE  = hits[hits == True].index.tolist()
GEPA_FALSE = hits[hits == False].index.tolist()

print(len(GEPA_TRUE))
print(len(GEPA_FALSE))

In [ ]:
random.seed(42)

train_true  = random.sample(GEPA_TRUE,  k=round(len(GEPA_TRUE)  * 0.6))
train_false = random.sample(GEPA_FALSE, k=round(len(GEPA_FALSE) * 0.6))

eval_true  = [r for r in GEPA_TRUE  if r not in train_true]
eval_false = [r for r in GEPA_FALSE if r not in train_false]

train = train_true + train_false
eval_ = eval_true  + eval_false

print(f"Train: {len(train)} ({len(train_true)} TRUE, {len(train_false)} FALSE)")
print(f"Eval:  {len(eval_)} ({len(eval_true)} TRUE, {len(eval_false)} FALSE)")

In [ ]:
sorted(train)

# GEPA Evaluation
vs. Thinking 1
##  (EXACT)

In [ ]:
_, gepa_eval_exact = eval(gs_extractions_raw_ynorm, True)

In [ ]:
gepa_eval_exact = gepa_eval_exact[['report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label', 'value_t1', 'value_g', 'unit_t1', 'unit_g', 'label_t1', 'label_g', 't1_value_hit','g_value_hit','t1_unit_hit','g_unit_hit']]

In [ ]:
gepa_improvement_exact = (gepa_eval_exact["t1_value_hit"] == False) & (gepa_eval_exact["g_value_hit"] == True)
gepa_eval_exact.loc[gepa_improvement_exact, ["report_name", "year", "scope", "value", "value_t1", "value_g"]]

## (ANY)

In [ ]:
_, gepa_eval = eval(gs_extractions_raw_ynorm, False)

In [ ]:
gepa_eval = gepa_eval[['report_name', 'year', 'scope', 'value', 'unit', 'unit_normalized', 'label', 'value_t1', 'value_g', 'unit_t1', 'unit_g', 'label_t1', 'label_g', 't1_value_hit','g_value_hit','t1_unit_hit','g_unit_hit']]

In [ ]:
gepa_improvement = (gepa_eval["t1_value_hit"] == False) & (gepa_eval["g_value_hit"] == True)

gepa_eval.loc[gepa_improvement, ["report_name", "year", "scope", "value", "value_t1", "value_g"]]


#### See, which reports profit from "any" matching, so where multiple values got extracted and therefore "exact" did not match.

In [ ]:
better_with_any = (gepa_eval["g_value_hit"] == True) & (gepa_eval_exact["g_value_hit"] == False)
gepa_eval.loc[better_with_any, ["report_name", "year", "scope", "value", "value_g"]]
